In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from fastai.vision.all import *
import torch
from mtrain.neg_mask.model.datasets.blur_pad_dl import (
    get_coords_for_set,
    get_center_crop_coords_from_mask, BlurPadInferDataset, blur_overwriter,
)
from mtrain.neg_mask.leveled_cropping import (
    create_crop_level_sample,
    make_crop_level_pairs_v2,
    load_crop_level_sample_from_directory,
)
from mtrain.neg_mask.model.predict.mk_tensors import (
    mk_8_chan,
    get_crop_pairs_from_full_image,
)
import cv2
import numpy as np
from mtrain.neg_mask.model.predict import core
from mtrain.neg_mask.crops import get_region_crops, Bbox
from mtrain.utils import DiskBooleanMask, DiskImage, show, overlay_mask_on_img as OV
import torch

In [ ]:
# # from mtrain.neg_mask.model.predict.core import reconstruct_probability_masks
# from mtrain.neg_mask.crops import padded_crop, bbox_only_mask
# import itertools
# from tqdm import tqdm
# from mtrain.neg_mask.model.datasets.blur_pad_dl import BlurPadDataset
# from torch.utils.data import DataLoader

# def reconstruct_probability_masks(image, mask, all_probs, bboxes):
#     """Map bbox predictions back to full image coordinates."""
#     num_labels = all_probs.shape[1]
#     h, w = mask.shape
#     res_masks = [np.zeros((h, w), dtype=np.float32) for i in range(num_labels)]
#     for i, bbox in enumerate(bboxes):
#         for label_idx in range(num_labels):
#             current_prob = all_probs[i, label_idx].item()
#             bbox_mask = mask[bbox.y : bbox.y2, bbox.x : bbox.x2].astype(bool)
#             res_masks[label_idx][bbox.y : bbox.y2, bbox.x : bbox.x2][bbox_mask] = (
#                 current_prob
#             )

#     return res_masks

# def predict_and_return_prob_masks(
#     image,
#     mask,
#     learner,
#     crop_size=130,
#     device=None,
#     blur_kernel_sz=5,
#     blur_sigma=3,
#     bbox_pad=5,
# ):
#     crops, masks, bboxes, inner_bboxes = get_crops_masks_bboxes(image, mask, crop_size, bbox_pad)
#     ds = BlurPadInferDataset(crops, masks, inner_bboxes, crop_size, blur_overwriter(blur_kernel_sz, blur_sigma))
#     dl = DataLoader(ds, 4)
#     learner.eval()
#     with torch.no_grad():
#         res = [
#             learner.model(b).softmax(dim=1)
#             for b in dl
#         ]
#         probs = torch.stack(list(itertools.chain.from_iterable(res)))
#         other_mask, trash_mask = reconstruct_probability_masks(image, mask, probs, bboxes)
#         return other_mask, trash_mask



# def get_crops_masks_bboxes(image, mask, crop_size, bbox_pad):
#     bboxes = list(get_region_crops(mask))
#     crops, masks, result_bboxes = [], [], []
#     for bbox in tqdm(bboxes):
#         tight_img, new_y1, new_x1 = padded_crop(image, bbox, crop_size)
#         tight_mask = bbox_only_mask(mask, bbox, crop_size)
#         inner_bbox = Bbox(bbox.x - new_x1, bbox.y - new_y1, bbox.w, bbox.h)
#         crops.append(tight_img)
#         masks.append(tight_mask)
#         result_bboxes.append(inner_bbox)
#     return crops, masks, bboxes, result_bboxes


# def get_padded_bbox_mask(mask, bbox, padding=10):
#     # 1. Find the coordinates of all non-zero pixels
#     x, y, w, h = bbox.x, bbox.y, bbox.w, bbox.h
#     img_h, img_w = mask.shape[:2]

#     # 3. Apply padding with boundary constraints
#     x1 = max(0, x - padding)
#     y1 = max(0, y - padding)
#     x2 = min(img_w, x + w + padding)
#     y2 = min(img_h, y + h + padding)

#     # 4. Create the new mask
#     padded_mask = np.zeros_like(mask)
#     cv2.rectangle(padded_mask, (x1, y1), (x2, y2), 255, -1)

#     return padded_mask, (x1, y1, x2, y2)


# def get_learner():
#     train_ds = BlurPadDataset([], Path("./masks"), 130, False, max_noise=None)
#     valid_ds = BlurPadDataset([], Path("./masks"), 130, True, max_noise=None)
#     dls = DataLoaders.from_dsets(
#         train_ds,
#         valid_ds,
#         device=default_device(),
#         num_workers=4,
#         bs=16,
#         # pin_memory=True,
#         persistent_workers=True,
#     )  # don't respawn workers each epoch)
#     learn = vision_learner(
#         dls,
#         resnet18,
#         metrics=[Precision()],
#         loss_func=CrossEntropyLossFlat(),
#         n_out=2,
#         normalize=False,
#         n_in=3,
#     )
#     learn = learn.remove_cb(ProgressCallback)
#     return learn

In [ ]:
from mtrain.example_dir.core import NEG_MASK_UNBLURRED_MODEL_PATH
from mtrain.example_dir import get_default_negmask_learner

learner = get_default_negmask_learner(NEG_MASK_UNBLURRED_MODEL_PATH)


In [ ]:
# direc = Path('/Users/hariomnarang/Desktop/personal/roads/datasets/inference/test_set/1634936550227999')
# direc = Path('/Users/hariomnarang/Desktop/personal/roads/datasets/inference/test_set/1217095282865379')
# direc = Path('/Users/hariomnarang/Desktop/personal/roads/datasets/inference/test_set/1634936550227999')
direc = Path('/Users/hariomnarang/Desktop/personal/roads/datasets/inference/test_set/291967965907673')

In [ ]:
image, mask = direc / "image.jpg", direc / "m2.png"
image, mask = DiskImage.load(image), DiskBooleanMask.load(mask)

# crops, masks, bboxes = get_crops_masks_bboxes(image, mask, 130, 10)
# image = cv2.resize(image, (1024,1024), interpolation=cv2.INTER_AREA)
# # mask = cv2.resize(mask, (1024,1024), interpolation=cv2.INTER_NEAREST)
# other_mask, trash_mask = predict_and_return_prob_masks(image, mask, learner, blur_kernel_sz=13, blur_sigma=4, bbox_pad=10)

show([image, mask])

In [ ]:
from mtrain.example_dir.core import NEGMASK_BLUR_KERNEL_SIGMA, NEGMASK_BLUR_KERNEL_SZ
from mtrain.example_dir.core import NEGMASK_SIZE, NEGMASK_BBOX_PAD
from mtrain.neg_mask.model.predict.trash import get_crops_masks_bboxes
crops, masks, bboxes, inner_bboxes = get_crops_masks_bboxes(
    image, mask, NEGMASK_SIZE, NEGMASK_BBOX_PAD
)
ds = BlurPadInferDataset(
    crops,
    masks,
    inner_bboxes,
    NEGMASK_SIZE,
    blur_overwriter(NEGMASK_BLUR_KERNEL_SZ, NEGMASK_BLUR_KERNEL_SIGMA),
)

In [ ]:
len(crops)

In [ ]:
idx = -1

In [ ]:
from mtrain.denorm import denormalize_imagenet
idx += 1
print(idx)
tens = ds[idx]
img = denormalize_imagenet(tens).permute([1,2,0]).numpy()
with torch.no_grad():
    vals = learner.model(ds[idx].unsqueeze(0))
other_prob, trash_prob = vals[0][0], vals[0][1]
print("other", other_prob.item(), "trash", trash_prob.item())
print("pred", "trash" if trash_prob > other_prob else "other")
show([img, crops[idx]])
# plt.imshow(img)

# idx += 1

# print("tight bbox in original", bboxes[idx])
# print("tight bbox in crop", inner_bboxes[idx])
# show([
#     crops[idx], masks[idx]
# ])

In [ ]:
from mtrain.neg_mask.model.gradcam import show_gradcam_for_image
viz, img = show_gradcam_for_image(learner, ds[26].unsqueeze(0), 1)

show([viz, img])

In [ ]:
learner.model(ds[idx].unsqueeze(0))